In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from flask_cors import CORS

In [ ]:
from flask import Flask, request, jsonify

## Import Pickle

In [ ]:
import pickle

In [ ]:
file = open('rain_model.pkl', 'rb')
rain_model = pickle.load(file)
file.close()

In [ ]:
file = open('wind_model.pkl', 'rb')
wind_model = pickle.load(file)
file.close()

In [ ]:
file = open('temp_model.pkl', 'rb')
temp_model = pickle.load(file)
file.close()

In [ ]:
file = open('daily_humidity.pkl', 'rb')
daily_humidity = pickle.load(file)
file.close()

file = open('daily_pressure.pkl', 'rb')
daily_pressure = pickle.load(file)
file.close()

file = open('daily_temp.pkl', 'rb')
daily_temp = pickle.load(file)
file.close()

file = open('daily_wind_speed.pkl', 'rb')
daily_wind_speed = pickle.load(file)
file.close()

file = open('daily_longwave.pkl', 'rb')
daily_longwave = pickle.load(file)
file.close()

file = open('daily_shortwave.pkl', 'rb')
daily_shortwave = pickle.load(file)
file.close()

file = open('daily_precip.pkl'  , 'rb')
daily_precip = pickle.load(file)
file.close()

file = open('daily_wind_speed.pkl', 'rb')
daily_wind_speed = pickle.load(file)
file.close()




In [ ]:
#Define a function that takes two dates and produces an array of dates between them
def split_dates(start, end):
    dates = pd.date_range(start, end)
    return dates

## Get/Post

In [ ]:
app = Flask(__name__)
CORS(app, origins=["http://localhost:5173"])

### Here

In [ ]:
def get_prediction(date, latitude, longitude):
    
        month = date.month
        day = date.day
        year = date.year
        hour = date.hour
        
      
        rain_df = {        
        'Humidity':[daily_humidity['Humidity'][month*30 +day]],
        'Pressure (Pa)': [daily_pressure['Pressure (Pa)'][month*30 +day]],
        'Downward Shortwave Radiation (W/m^2)' : [daily_shortwave['Downward Shortwave Radiation (W/m^2)'][month*30 + day]],
        'Downward Longwave Radiation (W/m^2)' : [daily_longwave['Downward Longwave Radiation (W/m^2)'][month*30 + day]],
        'Year':[year],
        'Month':[month],
        'Day':[day],
        'Hour':[hour],
        'Longitude':[longitude],
        'Latitude':[latitude]}
    
        wind_df = {        
        'Surface Air Pressure (Pa)' : [daily_pressure['Pressure (Pa)'][month * 30 + day]],
        'Surface Air Temp (K)': [daily_temp['Surface Air Temp (K)'][month * 30 + day]],
        'Year':[year],
        'Month':[month],
        'Day':[day],
        'Hour':[hour],
        'Longitude':[longitude],
        'Latitude':[latitude]}
   
        temp_df = {
        'Surface Wind Speed (m/s)':[daily_wind_speed['Surface Wind Speed (m/s)'][month * 30 + day]],
        'Humidity (g/kg)': [daily_humidity['Humidity'][month * 30 + day]],
        'Year':[year],
        'Month':[month],
        'Day':[day],
        'Hour':[hour],
        'Longitude':[longitude],
        'Latitude':[latitude] }

        rain_df1 = pd.DataFrame(rain_df)
        wind_df = pd.DataFrame(wind_df)
        temp_df = pd.DataFrame(temp_df)

        #convert rain into mm/hr
        rain = rain_model.predict(rain_df1)[0]
        rain = rain * 3600
        #convert wind into km/hr
        wind = wind_model.predict(wind_df)[0]
        wind = wind * 3.6
        #convert temperature into Celsius
        temp = temp_model.predict(temp_df)[0]
        temp = temp - 273.15



        prediction = {'Date': date, 'Rain': rain, 'Wind':wind, 'Temp':temp}
        return prediction

## Method 1 Range of Date

In [ ]:
@app.route("/request", methods=["POST"])
def predict2():

    data = request.get_json()

    latitude = data['Latitude']
    longitude = data['Longitude']
    
    start_date = data['Start_Date']
    end_date = data['End_Date']
    date_list = split_dates(start_date, end_date)

    pred_list = []
    for each in date_list:
        prediction = get_prediction(each, latitude, longitude)
        pred_list.append(prediction)
    
    print(pred_list)
    return jsonify(pred_list)

In [15]:
if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Oct/2025 20:15:47] "POST /request HTTP/1.1" 200 -


[{'Date': Timestamp('2025-10-01 08:00:00'), 'Rain': 0.09908557005814693, 'Wind': 13.694607770010817, 'Temp': 13.769319405286069}, {'Date': Timestamp('2025-10-02 08:00:00'), 'Rain': 0.10347677060158608, 'Wind': 14.020485137468098, 'Temp': 13.412251450404938}, {'Date': Timestamp('2025-10-03 08:00:00'), 'Rain': 0.09832810424914834, 'Wind': 13.968014445759644, 'Temp': 13.806283937491457}, {'Date': Timestamp('2025-10-04 08:00:00'), 'Rain': 0.11132059467783206, 'Wind': 14.263436465271798, 'Temp': 14.725688592851611}, {'Date': Timestamp('2025-10-05 08:00:00'), 'Rain': 0.10957807365210052, 'Wind': 14.200405100516377, 'Temp': 14.088628365373097}, {'Date': Timestamp('2025-10-06 08:00:00'), 'Rain': 0.08511638862233116, 'Wind': 13.286989301169546, 'Temp': 12.861478724113454}]


127.0.0.1 - - [05/Oct/2025 20:37:25] "OPTIONS /request HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2025 20:37:26] "POST /request HTTP/1.1" 200 -


[{'Date': Timestamp('2025-10-06 04:00:00'), 'Rain': 3.444670384721883e-05, 'Wind': 78033.91779323542, 'Temp': 206.02281707499088}, {'Date': Timestamp('2025-10-07 04:00:00'), 'Rain': 3.455165355614872e-05, 'Wind': 78033.29768195344, 'Temp': 206.01514665620607}, {'Date': Timestamp('2025-10-08 04:00:00'), 'Rain': 3.465660326507861e-05, 'Wind': 78032.67757067145, 'Temp': 206.0074762374212}, {'Date': Timestamp('2025-10-09 04:00:00'), 'Rain': 3.4761552974008503e-05, 'Wind': 78032.05745938944, 'Temp': 205.99980581863633}, {'Date': Timestamp('2025-10-10 04:00:00'), 'Rain': 3.4866502682938394e-05, 'Wind': 78031.43734810746, 'Temp': 205.99213539985146}, {'Date': Timestamp('2025-10-11 04:00:00'), 'Rain': 3.4971452391868284e-05, 'Wind': 78030.81723682546, 'Temp': 205.98446498106665}, {'Date': Timestamp('2025-10-12 04:00:00'), 'Rain': 3.5076402100798175e-05, 'Wind': 78030.19712554346, 'Temp': 205.97679456228178}, {'Date': Timestamp('2025-10-13 04:00:00'), 'Rain': 3.5181351809728065e-05, 'Wind': 780

127.0.0.1 - - [05/Oct/2025 20:37:29] "POST /request HTTP/1.1" 200 -


[{'Date': Timestamp('2025-10-06 04:00:00'), 'Rain': 3.444670384721883e-05, 'Wind': 78033.91779323542, 'Temp': 206.02281707499088}, {'Date': Timestamp('2025-10-07 04:00:00'), 'Rain': 3.455165355614872e-05, 'Wind': 78033.29768195344, 'Temp': 206.01514665620607}, {'Date': Timestamp('2025-10-08 04:00:00'), 'Rain': 3.465660326507861e-05, 'Wind': 78032.67757067145, 'Temp': 206.0074762374212}, {'Date': Timestamp('2025-10-09 04:00:00'), 'Rain': 3.4761552974008503e-05, 'Wind': 78032.05745938944, 'Temp': 205.99980581863633}, {'Date': Timestamp('2025-10-10 04:00:00'), 'Rain': 3.4866502682938394e-05, 'Wind': 78031.43734810746, 'Temp': 205.99213539985146}, {'Date': Timestamp('2025-10-11 04:00:00'), 'Rain': 3.4971452391868284e-05, 'Wind': 78030.81723682546, 'Temp': 205.98446498106665}, {'Date': Timestamp('2025-10-12 04:00:00'), 'Rain': 3.5076402100798175e-05, 'Wind': 78030.19712554346, 'Temp': 205.97679456228178}, {'Date': Timestamp('2025-10-13 04:00:00'), 'Rain': 3.5181351809728065e-05, 'Wind': 780

127.0.0.1 - - [05/Oct/2025 20:39:18] "OPTIONS /request HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2025 20:39:19] "POST /request HTTP/1.1" 200 -


[{'Date': Timestamp('2025-12-07 05:00:00'), 'Rain': 2.726948938819361e-05, 'Wind': 78077.49999453299, 'Temp': 206.77076400972965}, {'Date': Timestamp('2025-12-08 05:00:00'), 'Rain': 2.73744390971235e-05, 'Wind': 78076.87988325101, 'Temp': 206.76309359094478}, {'Date': Timestamp('2025-12-09 05:00:00'), 'Rain': 2.7479388806053392e-05, 'Wind': 78076.25977196901, 'Temp': 206.7554231721599}, {'Date': Timestamp('2025-12-10 05:00:00'), 'Rain': 2.7584338514983282e-05, 'Wind': 78075.63966068701, 'Temp': 206.74775275337504}, {'Date': Timestamp('2025-12-11 05:00:00'), 'Rain': 2.7689288223913173e-05, 'Wind': 78075.01954940501, 'Temp': 206.74008233459023}, {'Date': Timestamp('2025-12-12 05:00:00'), 'Rain': 2.7794237932843063e-05, 'Wind': 78074.39943812303, 'Temp': 206.73241191580536}, {'Date': Timestamp('2025-12-13 05:00:00'), 'Rain': 2.7899187641772954e-05, 'Wind': 78073.77932684103, 'Temp': 206.7247414970205}, {'Date': Timestamp('2025-12-14 05:00:00'), 'Rain': 2.8004137350702844e-05, 'Wind': 7807

In [ ]:
# ## Example 
# date1 = '2023-01-01'
# date2 = '2023-01-03'
# date_list = split_dates(date1, date2)

# pred_list = []
# for each in date_list:
#         prediction = get_prediction(each, 50, -80)
#         pred_list.append(prediction)

# print(pred_list)

[{'Date': Timestamp('2023-01-01 00:00:00'), 'Rain': 1.861835249021603e-05, 'Wind': 99796.17080719717, 'Temp': 263.85566124802926}, {'Date': Timestamp('2023-01-02 00:00:00'), 'Rain': 1.872330219914592e-05, 'Wind': 99795.55069591517, 'Temp': 263.8479908292444}, {'Date': Timestamp('2023-01-03 00:00:00'), 'Rain': 1.882825190807581e-05, 'Wind': 99794.93058463317, 'Temp': 263.8403204104596}]


In [ ]:
# ## Post will have 
# # Longitude/Latitude
# # Hour/Day/Month/Year
# @app.route("/request", methods=["POST"])
# def predict1():

#     data = request.get_json()

#     latitude = data['Latitude']
#     longitude = data['Longitude']
    
#     start_date = data['Start_Date']
#     end_date = data['End_Date']
#     date_list = split_dates(start_date, end_date)

#     prediction_list = []
#     #for each Date, create a dataframe, and predict the rain wind and temp
#     for each in date_list:
#         month = data['Month']
#         day = data['Day']
#         year = data['Year']
#         hour = data['Hour']


#         rain_df = {
#         'Humidity':[humidity_avg[month -1]],
#         'Pressure (Pa)': [air_pressure_avg[month -1]],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[longitude],
#         'Latitude':[latitude]}
    
#         wind_df = {        
#         'Surface Air Temp (K)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[longitude],
#         'Latitude':[latitude]}
   
#         temp_df = {
#         'Surface Wind Speed (m/s)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[longitude],
#         'Latitude':[latitude] }

#         rain_df1 = pd.DataFrame(rain_df)
#         wind_df = pd.DataFrame(wind_df)
#         temp_df = pd.DataFrame(temp_df)

#         rain = rain_model.predict(rain_df1)[0]
#         wind = wind_model.predict(wind_df)[0]
#         temp = temp_model.predict(temp_df)[0]

#         prediction = {'Date': each, 'Rain': rain, 'Wind':wind, 'Temp':temp}
#         prediction_list.append(prediction)
#         #end loop

#     return jsonify({ 'Predictions' : prediction_list})




In [ ]:
# date1 = '2023-01-01T08:00:00'
# date2 = '2023-01-02T08:00:00'
# dates = split_dates(date1, date2)
# # print(dates)


DatetimeIndex(['2023-01-01 08:00:00', '2023-01-02 08:00:00'], dtype='datetime64[ns]', freq='D')


In [ ]:
# pred_list = []

# for each in dates:
#         month = each.month
#         day = each.day
#         year = each.year
#         hour = each.hour


#         rain_df = {
#         'Humidity':[0],
#         'Pressure (Pa)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[50],
#         'Latitude':[-80]}
    
#         wind_df = {        
#         'Surface Air Temp (K)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[50],
#         'Latitude':[-80]}
   
#         temp_df = {
#         'Surface Wind Speed (m/s)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[50],
#         'Latitude':[-80] }

#         rain_df1 = pd.DataFrame(rain_df)
#         wind_df = pd.DataFrame(wind_df)
#         temp_df = pd.DataFrame(temp_df)

#         # rain = rain_model.predict(rain_df1)[0]
#         wind = wind_model.predict(wind_df)[0]
#         temp = temp_model.predict(temp_df)[0]

#         prediction = {'Date': each, 'Rain': rain, 'Wind':wind, 'Temp':temp}
#         pred_list.append(prediction)
    
# print(pred_list)

[{'Date': Timestamp('2023-01-01 08:00:00'), 'Rain': 0.001246843873389896, 'Wind': 181143.9715257681, 'Temp': 492.55329476651434}, {'Date': Timestamp('2023-01-02 08:00:00'), 'Rain': 0.001246948823098826, 'Wind': 181143.3514144861, 'Temp': 492.5456243477295}]


In [ ]:
# for each in dates:
#     print(each.day)
#     print(each.month)
#     # print(each.year)
#     print(each.hour)

1
1
2023
0
2
1
2023
0
3
1
2023
0
4
1
2023
0
5
1
2023
0
6
1
2023
0
7
1
2023
0
8
1
2023
0
9
1
2023
0
10
1
2023
0
